[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_04_optimizers/task_3_optimizers.ipynb)

# Week 4 · Make the network learn

**Goal:** turn week 3's gradients into parameter updates, then explain the learning curves. We reuse its **2 → 3 → 1 network, four XOR samples and mean squared error**.

Complete three **TODO** cells and three questions. Implement SGD and momentum; Adam, checks and plots are supplied. RMSprop and mini-batches are optional. Use CPU locally or in Colab; no downloads, accounts, W&B or Kaggle submissions are needed. Keep your notebook for the test. In the Colab URL, replace `fiit-ba` with your GitHub username to open your fork.

*Adapted from the FIIT NSIETE course materials (vgg-fiit/NSIETE_2026).*

In [ ]:
import matplotlib.pyplot as plt
import torch

SEED = 42
DTYPE = torch.float64

## 1. A gradient tells us which way to step

For a parameter $\theta$, gradient $g$ and learning rate $\eta$, gradient descent uses **$\theta\leftarrow\theta-\eta g$**. The minus sign moves toward lower loss for a sufficiently small step.

Worked example: $J(w)=(w-3)^2$ at $w=0$ has gradient $-6$. Try two step sizes below. One reduces the loss; the other jumps past the minimum and increases it.

In [ ]:
weight = 0.
gradient = 2 * (weight - 3)
for lr in [0.1, 1.1]:
    updated = weight - lr * gradient
    print(f"lr={lr}: weight {weight} → {updated:.1f}, loss 9 → {(updated - 3) ** 2:.2f}")

## 2. The network we already know (given)

This is the completed week 3 framework. Read `parameters()`: it returns `(parameter, gradient)` pairs for each weight matrix and bias. An optimizer changes the parameters using those gradients.

Forward saves values for backward. The loss averages once; the layers propagate its gradients without dividing again.

In [ ]:
class Module:
    def __init__(self):
        self.modules = {}

    def add_module(self, module, name):
        self.modules[name] = module

    def __call__(self, X):
        return self.forward(X)

class Linear(Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = torch.randn(out_features, in_features, dtype=DTYPE) * 0.1
        self.b = torch.zeros(out_features, 1, dtype=DTYPE)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)

    def forward(self, X):
        self.X = X
        return self.W @ X + self.b

    def backward(self, dZ):
        self.dW = dZ @ self.X.T
        self.db = dZ.sum(dim=1, keepdim=True)
        return self.W.T @ dZ

class ReLU(Module):
    def forward(self, X):
        self.X = X
        return torch.clamp(X, min=0)

    def backward(self, dA):
        return dA * (self.X > 0)

class Sigmoid(Module):
    def forward(self, X):
        self.A = 1 / (1 + torch.exp(-X))
        return self.A

    def backward(self, dA):
        return dA * self.A * (1 - self.A)

class Model(Module):
    def forward(self, X):
        for layer in self.modules.values():
            X = layer(X)
        return X

    def backward(self, dA):
        for layer in reversed(self.modules.values()):
            dA = layer.backward(dA)
        return dA

    def parameters(self):
        pairs = []
        for layer in self.modules.values():
            if isinstance(layer, Linear):
                pairs.extend([(layer.W, layer.dW), (layer.b, layer.db)])
        return pairs

In [ ]:
class MSELoss:
    def forward(self, prediction, target):
        return ((prediction - target) ** 2).mean()

    def backward(self, prediction, target):
        return 2 * (prediction - target) / prediction.numel()

In [ ]:
def create_model():
    torch.manual_seed(SEED)
    model = Model()
    model.add_module(Linear(2, 3), "hidden")
    model.add_module(ReLU(), "relu")
    model.add_module(Linear(3, 1), "output")
    model.add_module(Sigmoid(), "sigmoid")
    return model

In [ ]:
X = torch.tensor([[-1., -1., 1., 1.], [-1., 1., -1., 1.]], dtype=DTYPE)
Y = torch.tensor([[0., 1., 1., 0.]], dtype=DTYPE)
criterion = MSELoss()

## 3. SGD: use the current gradient

**TODO 1:** subtract `self.lr * gradient` from each parameter. `-=` updates the tensor held by the model.

The update rule is called **SGD**. Here each step uses all four samples, so we are doing full-batch gradient descent. With randomly selected samples or mini-batches, the same rule becomes stochastic gradient descent.

In [ ]:
class SGD:
    def __init__(self, model, lr):
        self.model = model
        self.lr = lr

    def step(self):
        for parameter, gradient in self.model.parameters():
            ...  # TODO 1: parameter -= learning rate * gradient

## 4. Momentum: remember previous gradients

Keep a separate velocity $v$ for each parameter, initially zero:

$$v\leftarrow\beta v+g,\qquad \theta\leftarrow\theta-\eta v.$$

Persistent gradient directions build up; directions that alternate can cancel. **TODO 2:** update each velocity, then use it to update the parameter. Do not reset the velocities at each step. This is PyTorch's momentum convention: the new gradient has coefficient $1$.

In [ ]:
class Momentum(SGD):
    def __init__(self, model, lr, beta=0.9):
        super().__init__(model, lr)
        self.beta = beta
        self.velocity = [torch.zeros_like(p) for p, g in model.parameters()]

    def step(self):
        for i, (parameter, gradient) in enumerate(self.model.parameters()):
            self.velocity[i] = ...  # TODO 2: beta * old velocity + gradient
            ...  # TODO 2: parameter -= learning rate * velocity

## 5. Adam: track direction and scale (given)

Adam keeps a moving average $m$ of gradients and $v$ of squared gradients. It corrects their initial bias toward zero, then scales each parameter's update:

$$m\leftarrow\beta_1m+(1-\beta_1)g,\quad v\leftarrow\beta_2v+(1-\beta_2)g^2,$$
$$\hat m=\frac{m}{1-\beta_1^t},\quad \hat v=\frac{v}{1-\beta_2^t},\quad
\theta\leftarrow\theta-\eta\frac{\hat m}{\sqrt{\hat v}+\varepsilon}.$$

Trace the supplied code: `t` increases once per optimizer step; each parameter has its own `m` and `v`. The small `eps` prevents division by zero. These update rules use ordinary tensors, without autograd.

In [ ]:
class Adam(SGD):
    def __init__(self, model, lr, beta1=0.9, beta2=0.999, eps=1e-8):
        super().__init__(model, lr)
        self.beta1, self.beta2, self.eps = beta1, beta2, eps
        self.t = 0
        self.m = [torch.zeros_like(p) for p, g in model.parameters()]
        self.v = [torch.zeros_like(p) for p, g in model.parameters()]

    def step(self):
        self.t += 1
        for i, (parameter, gradient) in enumerate(self.model.parameters()):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * gradient
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * gradient ** 2
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            parameter -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

### Check the update rules (given)

Each optimizer receives the same parameters and the same made-up gradients as its PyTorch counterpart. Comparing **three steps** checks that momentum and Adam remember their state correctly. This isolates the update rule from the network's backward pass.

In [ ]:
def check_optimizer(make_ours, make_reference):
    model = create_model()
    reference = [p.clone() for p, g in model.parameters()]
    ours = make_ours(model)
    theirs = make_reference(reference)
    generator = torch.Generator().manual_seed(7)
    for step in range(3):
        for ref, (parameter, gradient) in zip(reference, model.parameters()):
            gradient.copy_(torch.randn(gradient.shape, generator=generator, dtype=DTYPE))
            ref.grad = gradient.clone()
        ours.step()
        theirs.step()
    for (parameter, gradient), ref in zip(model.parameters(), reference):
        torch.testing.assert_close(parameter, ref, atol=1e-10, rtol=0)
    print("OK:", type(ours).__name__, "matches torch.optim after 3 steps")

check_optimizer(lambda m: SGD(m, 0.1), lambda p: torch.optim.SGD(p, lr=0.1))
check_optimizer(lambda m: Momentum(m, 0.1), lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9))
check_optimizer(lambda m: Adam(m, 0.03), lambda p: torch.optim.Adam(p, lr=0.03))

## 6. Training repeats the same three steps

1. **Forward:** predict and measure the loss.
2. **Backward:** calculate gradients from the loss through the network.
3. **Update:** let the optimizer change the parameters.

**TODO 3:** get `criterion.backward(prediction, Y)`, pass it to `model.backward(...)`, then call `optimizer.step()`. A new forward pass is needed each step because the parameters changed. Our backward pass overwrites the stored gradients each time.

In [ ]:
def fit(model, optimizer, steps=1500):
    losses = [criterion.forward(model(X), Y).item()]
    for step in range(steps):
        prediction = model(X)
        d_prediction = ...  # TODO 3: gradient of the loss
        ...  # TODO 3: model backward
        ...  # TODO 3: optimizer step
        losses.append(criterion.forward(model(X), Y).item())
    return losses

**Run the comparison (given).** Each run resets the seed, so every optimizer starts with identical weights and sees the same four samples for 1,500 updates. Plot the **training loss** and inspect the final predictions.

The learning rates below are example settings. A shared starting point helps compare them, but one dataset and one learning rate per optimizer cannot establish a universal ranking. Fitting these four points demonstrates learning; it does not measure performance on unseen data.

In [ ]:
factories = {"SGD": lambda m: SGD(m, 0.1), "Momentum": lambda m: Momentum(m, 0.1), "Adam": lambda m: Adam(m, 0.03)}
results = {}
for name, make_optimizer in factories.items():
    model = create_model()
    results[name] = fit(model, make_optimizer(model))
    assert results[name][-1] < results[name][0], "Loss did not decrease: check TODO 3"
    print(name, "loss:", round(results[name][-1], 5), "predictions:", model(X).round(decimals=2))
    plt.plot(results[name], label=name)
plt.xlabel("parameter updates")
plt.ylabel("training MSE")
plt.legend()
plt.show()

### Change the learning rate

Run this supplied SGD experiment with `0.01`, `0.1` and `1.0`, keeping everything else fixed. Compare the loss curves with the scalar example in section 1. A learning rate that works for one problem need not work for another.

In [ ]:
for lr in [0.01, 0.1, 1.0]:
    model = create_model()
    losses = fit(model, SGD(model, lr))
    plt.plot(losses, label=f"lr={lr}")
plt.xlabel("parameter updates")
plt.ylabel("training MSE")
plt.legend()
plt.show()

## 7. Ready for the test?

1. Explain the roles of `criterion.backward`, `model.backward` and `optimizer.step`. Which one changes the parameters?

   _Your answer here._

2. Why can a small learning rate be slow and a large one overshoot? Use the worked example and your curves.

   _Your answer here._

3. What does momentum remember that SGD does not? What extra information does Adam keep?

   _Your answer here._

Restart and run all cells. The optimizer checks should print `OK`; inspect the loss curves and explain the updates. **No notebook, commit link or W&B link is required.**

## Optional: RMSprop and mini-batches

**RMSprop** keeps only an average of squared gradients and uses it to scale the current gradient:

$$s\leftarrow\alpha s+(1-\alpha)g^2,\qquad
\theta\leftarrow\theta-\eta\frac{g}{\sqrt{s}+\varepsilon}.$$

The supplied implementation is checked against PyTorch. Add it to `factories` with learning rate `0.01` to compare its curve.

**Mini-batches:** extend `fit` to shuffle the four samples and update on two at a time. Slice both `X` and `Y` along their columns. The loss already averages over the current batch. One pass through all samples is an *epoch*; smaller batches mean more updates per epoch, so compare curves using the same horizontal-axis definition.

In [ ]:
class RMSprop(SGD):
    def __init__(self, model, lr, alpha=0.9, eps=1e-8):
        super().__init__(model, lr)
        self.alpha, self.eps = alpha, eps
        self.square = [torch.zeros_like(p) for p, g in model.parameters()]

    def step(self):
        for i, (parameter, gradient) in enumerate(self.model.parameters()):
            self.square[i] = self.alpha * self.square[i] + (1 - self.alpha) * gradient ** 2
            parameter -= self.lr * gradient / (self.square[i].sqrt() + self.eps)

check_optimizer(lambda m: RMSprop(m, 0.01), lambda p: torch.optim.RMSprop(p, lr=0.01, alpha=0.9))